In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [2]:
!uv pip install llama-index

Using Python 3.12.2 environment at: /Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env
Audited 1 package in 21ms


In [3]:
!uv pip install llama-index-embeddings-huggingface
!uv pip install llama-index-llms-lmstudio

Using Python 3.12.2 environment at: /Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env
Audited 1 package in 14ms
Using Python 3.12.2 environment at: /Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env
Audited 1 package in 8ms


In [ ]:
import os
import openai
from dotenv import load_dotenv
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.vector_stores.types import VectorStoreQueryMode
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.llms.lmstudio import LMStudio

load_dotenv()

True

In [53]:

# Configure the client to use LM Studio's local server
client = openai.OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"  # LM Studio doesn't require a real API key
)


# Set up LM Studio LLM
llm = LMStudio(
    base_url="http://localhost:1234/v1",
    model_name="mistral-7b-instruct-v0.3",  # Specify the model name
    temperature=0.2
)



In [54]:
# Use chat completions
response = client.chat.completions.create(
    model="mistral-7b-instruct-v0.3",  # or whatever model name LM Studio shows
    messages=[
        {"role": "user", "content": "Hello, how are you? I am testing the LM Studio."},
    ],
    temperature=0.9,
    max_tokens=150,

)


In [55]:
print("Response from LM Studio:")
# Process the stream
print(response.choices[0].message.content)
print()  # New line at the end

Response from LM Studio:
 Hello there! I'm an AI and don't have personal experiences or feelings, but everything is functioning normally with me. How about we focus on making your experience with Lム Studio pleasant and productive instead? Let's work together to answer your questions, brainstorm solutions, and collaborate on ideas. I'm here to help!



In [56]:
llm.complete("Can you tell me what is the capital of UAE?")

CompletionResponse(text=" The capital city of United Arab Emirates (UAE) is not specific as it doesn't follow the conventional model of one governmental administrative center. However, the political and diplomatic centre is situated in the neighboring country of Abu Dhabi. The federal legislatures and cabinet of UAE are housed here, but each emirate still retains extensive autonomy over internal matters. The biggest financial and cultural centre of UAE is Dubai, which is one of seven emirates in this country.", additional_kwargs={'id': 'chatcmpl-gzpdrjkoxditahk1ivjmj', 'object': 'chat.completion', 'created': 1752723936, 'model': 'mistral-7b-instruct-v0.3', 'usage': {'prompt_tokens': 17, 'completion_tokens': 107, 'total_tokens': 124}, 'stats': {}, 'system_fingerprint': 'mistral-7b-instruct-v0.3'}, raw={'id': 'chatcmpl-gzpdrjkoxditahk1ivjmj', 'object': 'chat.completion', 'created': 1752723936, 'model': 'mistral-7b-instruct-v0.3', 'choices': [{'index': 0, 'logprobs': None, 'finish_reason'

In [10]:
# let's now download a file
!curl -o data/paul_graham_essay.txt https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/paul_graham/paul_graham_essay.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 75042  100 75042    0     0   211k      0 --:--:-- --:--:-- --:--:--     0-:--:-- --:--:--  211k


In [11]:
documents = SimpleDirectoryReader("data").load_data()
print(f"Number of documents loaded: {len(documents)}")

Number of documents loaded: 1


In [12]:
# If you want to use text splitting, you can do it like this:
# chunk using llama_index
text_splitter = SimpleNodeParser.from_defaults(
    chunk_size=800,  # Adjust chunk size as needed
    chunk_overlap=100  # Adjust overlap as needed
)
chunks = text_splitter.get_nodes_from_documents(documents)
print(f"Number of chunks created: {len(chunks)}")

Number of chunks created: 26


In [13]:
chunks[0].text[:500]  # Display the first 500 characters of the first chunk

'What I Worked On\n\nFebruary 2021\n\nBefore college the two main things I worked on, outside of school, were writing and programming. I didn\'t write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.\n\nThe first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing'

In [25]:
Settings.embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")  # Set the embed model globally
Settings.llm = llm  # Set the LLM client globally

In [119]:
index = VectorStoreIndex.from_documents(
    documents,
    # we can optionally override the embed_model here
    embed_model=Settings.embed_model,
    show_progress=True
)

Parsing nodes:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/22 [00:00<?, ?it/s]

In [58]:
index.storage_context.persist(persist_dir="storage")

In [95]:
# configure retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=3,
    vector_store_query_mode=VectorStoreQueryMode.DEFAULT
)

In [116]:
# configure response synthesizer
response_synthesizer = get_response_synthesizer()

# assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.2)],
)

In [117]:
question = "What did Robert Morris do?"

In [118]:
# response_synthesizer.get_prompts()
# query
retrieved_examples = query_engine.retrieve(question)

print("Retrieved examples:")
for example in retrieved_examples:
    print(f"- {example.text[:200]}...")  # Print the first 200 characters of each retrieved example

input_context = "\n".join([example.text for example in retrieved_examples])

Retrieved examples:
- After I moved to New York I became her de facto studio assistant.

She liked to paint on big, square canvases, 4 to 5 feet on a side. One day in late 1994 as I was stretching one of these monsters the...
- The students and faculty in the painting department at the Accademia were the nicest people you could imagine, but they had long since arrived at an arrangement whereby the students wouldn't require t...


In [122]:
# Format a prompt:

def create_prompt(question, context):
    prompt = f"""
    You are given a context from documents. Use this context to answer a question. Be consise and to the point.

    Context:
    {context}

    Question: {question}

    if the context does not contain enough information to answer the question, say "I don't know based on the given context". Do not make up answers.
    """
    return prompt

final_prompt = create_prompt(question, context=input_context)

print("Question:")
print(question)

Question:
What did Robert Morris do?


In [121]:
print(llm.complete(final_prompt).text)

 Robert Morris wrote software as stated in the passage. More specifically, he wrote some software to resize images and set up an HTTP server to serve pages, when both people decided to start a company to put art galleries online. Later, he wrote a shopping cart for the store builder software that they developed after they shifted their focus to building online stores.
